# Assignment: Alpha or Noise? Detecting Data Snooping in Strategy Selection
**Statistical Validation of Trading Strategies**

### Objective:
- Generate a **synthetic strategy dataset** (one candidate + 1,000 random benchmarks).
- Implement **7 performance metrics** from the presentation: Sharpe, Sortino, Calmar, Skewness, Kurtosis, Jarque-Bera, T-test p-value, and Bootstrap CI for Sharpe.
- Examine **metric sensitivity**: which metrics agree, which diverge, and what depth each adds.
- Run **White's Reality Check** to determine whether the strategy's Sharpe ratio beats the field by chance.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import plotly.subplots as ps
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── Simulation Parameters ─────────────────────────────────────────────────────
N_DAYS          = 252 * 3          # 3 years of daily returns
N_RANDOM_STRATS = 1_000            # pool size for White's Reality Check
ANNUAL_FACTOR   = np.sqrt(252)     # annualisation for Sharpe / Sortino
BOOTSTRAP_REPS  = 5_000            # bootstrap resamples for Sharpe CI

print(f"Simulation: {N_DAYS} trading days  |  {N_RANDOM_STRATS} benchmark strategies")

Simulation: 756 trading days  |  1000 benchmark strategies


## 1  |  Synthetic Data Generation

We create one **candidate strategy** with a modest positive drift and a pool of **1,000 purely random strategies** (zero drift, same volatility). If the candidate looks good only because we cherry-picked it from a large pool, White's Reality Check will expose that.

In [2]:
# ── Candidate strategy: slight positive drift ─────────────────────────────────
MU_CANDIDATE  = 0.0003   # ~7.5 % annualised drift
VOL_DAILY     = 0.010    # ~16 % annualised vol

candidate_returns = rng.normal(MU_CANDIDATE, VOL_DAILY, N_DAYS)

# ── Random benchmark pool: pure noise ────────────────────────────────────────
random_pool = rng.normal(0.0, VOL_DAILY, (N_RANDOM_STRATS, N_DAYS))

print(f"Candidate mean daily return : {candidate_returns.mean():.5f}")
print(f"Random pool mean daily return: {random_pool.mean():.5f}")

Candidate mean daily return : -0.00010
Random pool mean daily return: -0.00001


## 2  |  Performance Metrics

We implement **7 metrics** from the presentation:

| # | Metric | What it captures |
|---|--------|------------------|
| 1 | **Sharpe Ratio** | Risk-adjusted return (mean / std) |
| 2 | **Sortino Ratio** | Downside risk only |
| 3 | **Calmar Ratio** | Return vs. maximum drawdown |
| 4 | **Skewness** | Asymmetry of return distribution |
| 5 | **Kurtosis** | Fat-tail risk |
| 6 | **Jarque-Bera p-value** | Normality test |
| 7 | **T-test p-value** | Is mean return significantly > 0? |
| 8 | **Bootstrap CI (Sharpe)** | Uncertainty around Sharpe estimate |

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Helper: maximum drawdown from a return series
# ─────────────────────────────────────────────────────────────────────────────
def max_drawdown(returns: np.ndarray) -> float:
    cum = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(cum)
    dd = (cum - running_max) / running_max
    return dd.min()          # negative number; most negative = worst drawdown


# ─────────────────────────────────────────────────────────────────────────────
# Helper: bootstrap Sharpe CI
# ─────────────────────────────────────────────────────────────────────────────
def bootstrap_sharpe_ci(returns: np.ndarray,
                         n_reps: int = BOOTSTRAP_REPS,
                         ci: float = 0.95,
                         rng_obj=None) -> tuple[float, float]:
    """Return (lower, upper) bootstrap confidence interval for annualised Sharpe."""
    if rng_obj is None:
        rng_obj = np.random.default_rng(SEED)
    n = len(returns)
    sharpes = np.empty(n_reps)
    for i in range(n_reps):
        sample = rng_obj.choice(returns, size=n, replace=True)
        sharpes[i] = (sample.mean() / sample.std(ddof=1)) * ANNUAL_FACTOR
    alpha = (1 - ci) / 2
    return float(np.quantile(sharpes, alpha)), float(np.quantile(sharpes, 1 - alpha))


# ─────────────────────────────────────────────────────────────────────────────
# Main metrics function
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(returns: np.ndarray, run_bootstrap: bool = False) -> dict:
    mu   = returns.mean()
    sig  = returns.std(ddof=1)
    
    # --- 1. Sharpe
    sharpe = (mu / sig) * ANNUAL_FACTOR if sig > 0 else np.nan
    
    # --- 2. Sortino  (downside std only)
    downside = returns[returns < 0]
    dsig = downside.std(ddof=1) if len(downside) > 1 else np.nan
    sortino = (mu / dsig) * ANNUAL_FACTOR if dsig and dsig > 0 else np.nan
    
    # --- 3. Calmar  (annualised return / |max drawdown|)
    ann_ret = mu * 252
    mdd = max_drawdown(returns)
    calmar = ann_ret / abs(mdd) if mdd != 0 else np.nan
    
    # --- 4. Skewness
    skew = stats.skew(returns)
    
    # --- 5. Excess Kurtosis  (normal = 0 in scipy's convention)
    kurt = stats.kurtosis(returns)
    
    # --- 6. Jarque-Bera p-value
    _, jb_pval = stats.jarque_bera(returns)
    
    # --- 7. T-test p-value (one-sided: H1: mean > 0)
    t_stat, t_pval_two = stats.ttest_1samp(returns, popmean=0)
    t_pval_one = t_pval_two / 2 if t_stat > 0 else 1 - t_pval_two / 2
    
    result = dict(
        sharpe=sharpe,
        sortino=sortino,
        calmar=calmar,
        skewness=skew,
        kurtosis=kurt,
        jb_pval=jb_pval,
        t_pval=t_pval_one,
        ann_return=ann_ret,
        max_dd=mdd,
    )
    
    # --- 8. Bootstrap CI for Sharpe (optional – slow for 1k strategies)
    if run_bootstrap:
        lo, hi = bootstrap_sharpe_ci(returns)
        result['sharpe_ci_lo'] = lo
        result['sharpe_ci_hi'] = hi
    
    return result


# Compute for candidate
candidate_metrics = compute_metrics(candidate_returns, run_bootstrap=True)

print("\n── Candidate Strategy Metrics ──────────────────────────────────")
for k, v in candidate_metrics.items():
    print(f"  {k:<18}: {v:+.4f}")


── Candidate Strategy Metrics ──────────────────────────────────
  sharpe            : -0.1626
  sortino           : -0.2709
  calmar            : -0.0955
  skewness          : +0.0320
  kurtosis          : -0.0865
  jb_pval           : +0.8333
  t_pval            : +0.6108
  ann_return        : -0.0255
  max_dd            : -0.2670
  sharpe_ci_lo      : -1.3193
  sharpe_ci_hi      : +0.9672


## 3  |  Equity Curve & Drawdown

In [4]:
cum_pnl = np.cumprod(1 + candidate_returns)
running_max = np.maximum.accumulate(cum_pnl)
drawdown_series = (cum_pnl - running_max) / running_max
days = np.arange(N_DAYS)

fig = ps.make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=["Candidate Strategy – Equity Curve", "Drawdown"]
)

fig.add_trace(go.Scatter(
    x=days, y=cum_pnl,
    mode='lines', name='Equity',
    line=dict(color='#00d4aa', width=1.5)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=days, y=drawdown_series,
    mode='lines', name='Drawdown',
    fill='tozeroy',
    line=dict(color='#ff6b6b', width=1),
    fillcolor='rgba(255,107,107,0.25)'
), row=2, col=1)

fig.update_layout(
    template='plotly_dark',
    height=500,
    showlegend=True,
    margin=dict(t=50, b=40)
)
fig.show()

## 4  |  Return Distribution Analysis

Overlaying the candidate's return histogram against a fitted normal helps visualise the fat-tail / skew characteristics captured by kurtosis, skewness, and JB.

In [5]:
x_range = np.linspace(candidate_returns.min(), candidate_returns.max(), 300)
normal_pdf = stats.norm.pdf(x_range, candidate_returns.mean(), candidate_returns.std())

m = candidate_metrics

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=candidate_returns,
    nbinsx=60,
    histnorm='probability density',
    name='Observed returns',
    marker_color='rgba(0,212,170,0.55)',
    marker_line=dict(color='#00d4aa', width=0.4)
))

fig.add_trace(go.Scatter(
    x=x_range, y=normal_pdf,
    mode='lines', name='Fitted Normal',
    line=dict(color='lightsalmon', width=2, dash='dash')
))

annotation_text = (
    f"Skewness: {m['skewness']:+.3f}<br>"
    f"Ex. Kurtosis: {m['kurtosis']:+.3f}<br>"
    f"JB p-value: {m['jb_pval']:.4f}<br>"
    f"T-test p (>0): {m['t_pval']:.4f}"
)

fig.add_annotation(
    x=0.97, y=0.97,
    xref='paper', yref='paper',
    text=annotation_text,
    showarrow=False,
    align='right',
    bgcolor='rgba(0,0,0,0.6)',
    bordercolor='#555',
    font=dict(size=12, color='white')
)

fig.update_layout(
    title='Return Distribution vs. Normal',
    xaxis_title='Daily Return',
    yaxis_title='Density',
    template='plotly_dark',
    legend=dict(x=0.02, y=0.97)
)
fig.show()

## 5  |  Bootstrap Confidence Interval for Sharpe

A point estimate of the Sharpe ratio can be misleading. Bootstrapping the statistic shows the **uncertainty** around the estimate — crucial when deciding whether a strategy is truly positive-Sharpe.

In [6]:
# Re-collect all bootstrap Sharpe samples for the candidate
boot_rng = np.random.default_rng(SEED)
n = len(candidate_returns)
boot_sharpes = np.array([
    (boot_rng.choice(candidate_returns, size=n, replace=True).mean() /
     boot_rng.choice(candidate_returns, size=n, replace=True).std(ddof=1)) * ANNUAL_FACTOR
    for _ in range(BOOTSTRAP_REPS)
])

ci_lo = candidate_metrics['sharpe_ci_lo']
ci_hi = candidate_metrics['sharpe_ci_hi']
point_sharpe = candidate_metrics['sharpe']

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=boot_sharpes,
    nbinsx=80,
    name='Bootstrap Sharpe',
    marker_color='rgba(100,149,237,0.6)',
    marker_line=dict(color='cornflowerblue', width=0.3)
))

for x_val, label, color in [
    (point_sharpe, f'Point Sharpe = {point_sharpe:.2f}', '#00d4aa'),
    (ci_lo, f'95% CI lower = {ci_lo:.2f}', '#ff6b6b'),
    (ci_hi, f'95% CI upper = {ci_hi:.2f}', '#ff6b6b'),
    (0, 'Sharpe = 0', 'white'),
]:
    fig.add_vline(x=x_val, line_dash='dash', line_color=color,
                  annotation_text=label, annotation_position='top')

fig.update_layout(
    title='Bootstrap Distribution of Annualised Sharpe Ratio',
    xaxis_title='Sharpe Ratio',
    yaxis_title='Count',
    template='plotly_dark',
    showlegend=False
)
fig.show()

print(f"\nPoint Sharpe : {point_sharpe:.3f}")
print(f"95% CI       : [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"CI includes 0: {ci_lo <= 0 <= ci_hi}  ← genuine concern if True")


Point Sharpe : -0.163
95% CI       : [-1.319, 0.967]
CI includes 0: True  ← genuine concern if True


## 6  |  Metric Sensitivity Analysis

We compute all metrics across the **random pool** and examine correlations. Metrics that rank strategies similarly are redundant; metrics that diverge reveal **different dimensions of risk**.

In [7]:
print("Computing metrics for 1,000 random strategies ... (may take ~15 s)")

pool_records = []
for i, row in enumerate(random_pool):
    m = compute_metrics(row, run_bootstrap=False)
    pool_records.append(m)

pool_df = pd.DataFrame(pool_records)

# Add candidate as a named row
cand_row = {k: v for k, v in candidate_metrics.items()
            if k not in ('sharpe_ci_lo', 'sharpe_ci_hi')}
all_df = pd.concat([pool_df, pd.DataFrame([cand_row])], ignore_index=True)

print(f"Pool metrics computed. Shape: {pool_df.shape}")
pool_df[['sharpe', 'sortino', 'calmar', 'skewness', 'kurtosis', 'jb_pval', 't_pval']].describe().round(3)

Computing metrics for 1,000 random strategies ... (may take ~15 s)
Pool metrics computed. Shape: (1000, 9)


,sharpe,sortino,calmar,skewness,kurtosis,jb_pval,t_pval
count,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000
mean,-0.011,-0.008,0.100,0.000,0.005,0.498,0.508
std,0.579,0.962,0.426,0.093,0.180,0.293,0.287
min,-1.900,-3.002,-0.491,-0.247,-0.478,0.000,0.000
25%,-0.413,-0.689,-0.192,-0.065,-0.118,0.251,0.269
50%,-0.015,-0.025,-0.011,-0.002,-0.014,0.494,0.510
75%,0.356,0.584,0.255,0.061,0.112,0.756,0.763
max,2.004,3.453,3.853,0.273,0.744,1.000,0.999


In [8]:
# ── Rank correlations between metrics ────────────────────────────────────────
METRIC_COLS = ['sharpe', 'sortino', 'calmar', 'skewness', 'kurtosis', 'jb_pval', 't_pval']

corr = pool_df[METRIC_COLS].corr(method='spearman')

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=METRIC_COLS,
    y=METRIC_COLS,
    colorscale='RdBu',
    zmid=0,
    text=np.round(corr.values, 2),
    texttemplate='%{text}',
    showscale=True
))

fig.update_layout(
    title='Spearman Rank Correlation Between Metrics (random pool)',
    template='plotly_dark',
    height=500,
    width=620
)
fig.show()

In [9]:
# ── Scatter: Sharpe vs. Sortino  &  Sharpe vs. Calmar ────────────────────────
fig = ps.make_subplots(rows=1, cols=2,
                        subplot_titles=['Sharpe vs. Sortino', 'Sharpe vs. Calmar'])

for col_idx, (y_col, y_label) in enumerate([('sortino', 'Sortino'), ('calmar', 'Calmar')], start=1):
    fig.add_trace(go.Scatter(
        x=pool_df['sharpe'], y=pool_df[y_col],
        mode='markers',
        marker=dict(size=3, color='rgba(100,149,237,0.4)'),
        name=f'Random ({y_label})'
    ), row=1, col=col_idx)
    
    # Highlight candidate
    fig.add_trace(go.Scatter(
        x=[cand_row['sharpe']], y=[cand_row[y_col]],
        mode='markers',
        marker=dict(size=10, color='#00d4aa', symbol='star'),
        name='Candidate'
    ), row=1, col=col_idx)

fig.update_xaxes(title_text='Sharpe')
fig.update_layout(
    template='plotly_dark',
    title='Return-Based Metrics: How Consistently Do They Rank?',
    height=400,
    showlegend=False
)
fig.show()

In [10]:
# ── Rankings by metric: where does the candidate land? ───────────────────────
cand_percentiles = {}
for col in METRIC_COLS:
    cand_val   = cand_row[col]
    pool_vals  = pool_df[col].dropna()
    # For p-values lower = better, so invert
    if col in ('jb_pval', 't_pval'):
        pct = 100 * (pool_vals > cand_val).mean()   # % of random that are worse (higher p-val)
    else:
        pct = 100 * (pool_vals < cand_val).mean()   # % of random that are below candidate
    cand_percentiles[col] = pct

pct_df = pd.Series(cand_percentiles, name='Candidate Percentile (%)').round(1)
print("Candidate percentile rank within random pool:")
print(pct_df.to_string())

fig = go.Figure(go.Bar(
    x=pct_df.index,
    y=pct_df.values,
    marker_color=['#00d4aa' if v >= 90 else
                  ('lightsalmon' if v >= 70 else '#ff6b6b') for v in pct_df.values],
    text=[f"{v:.0f}%" for v in pct_df.values],
    textposition='outside'
))

fig.add_hline(y=90, line_dash='dash', line_color='white', opacity=0.4,
              annotation_text='90th pct threshold')

fig.update_layout(
    title="Candidate Strategy Percentile Rank Across Metrics",
    yaxis=dict(title='Percentile vs. random pool', range=[0, 105]),
    xaxis_title='Metric',
    template='plotly_dark'
)
fig.show()

Candidate percentile rank within random pool:
sharpe      39.0
sortino     39.1
calmar      38.3
skewness    63.6
kurtosis    31.8
jb_pval     17.3
t_pval      39.0


## 7  |  White's Reality Check

**Intuition:** If we search over $N$ strategies and pick the best, the best will look good *purely by luck*. White's Reality Check corrects for this multiple-comparisons problem by asking:

> *What is the probability that the **best** strategy in a random pool beats the candidate by chance?*

### Algorithm (stationary bootstrap approximation)
1. Let $f_k = \hat{S}_k - \hat{S}_0$ = Sharpe of strategy $k$ minus the candidate's Sharpe.
2. The WRC statistic is $V = \max_k f_k$.
3. Bootstrap $B$ resamples of the returns; recompute $V^*_b$ under each.
4. **p-value** = fraction of $V^*_b \geq V$.

In [11]:
def sharpe_ratio(ret: np.ndarray) -> float:
    s = ret.std(ddof=1)
    return (ret.mean() / s) * ANNUAL_FACTOR if s > 0 else 0.0


def whites_reality_check(candidate: np.ndarray,
                          pool: np.ndarray,
                          n_boot: int = 1_000,
                          rng_obj=None) -> dict:
    """
    White (2000) Reality Check.

    Parameters
    ----------
    candidate : shape (T,)   – returns of the strategy being evaluated
    pool      : shape (K, T) – returns of K competing (benchmark) strategies
    n_boot    : number of bootstrap resamples

    Returns
    -------
    dict with keys: wrc_pvalue, best_pool_sharpe, candidate_sharpe, V_observed
    """
    if rng_obj is None:
        rng_obj = np.random.default_rng(SEED)

    T = len(candidate)
    K = len(pool)

    cand_sharpe = sharpe_ratio(candidate)

    # Excess performance of each pool strategy over candidate (on raw returns)
    # f_k = Sharpe(pool_k) - Sharpe(candidate)  — computed on *observed* data
    pool_sharpes = np.array([sharpe_ratio(pool[k]) for k in range(K)])
    f_observed   = pool_sharpes - cand_sharpe      # shape (K,)
    V_observed   = f_observed.max()                # the WRC test statistic

    # Bootstrap distribution of V under the null (H0: E[f_k] <= 0 for all k)
    V_boot = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng_obj.integers(0, T, size=T)        # resample indices
        cand_b  = sharpe_ratio(candidate[idx])
        pool_b  = np.array([sharpe_ratio(pool[k][idx]) for k in range(K)])
        f_boot  = pool_b - cand_b
        # Demeaned to enforce H0: subtract observed f so E*[f*] = f_observed
        f_boot_demeaned = f_boot - np.maximum(f_observed, 0)
        V_boot[b] = f_boot_demeaned.max()

    wrc_pvalue = float((V_boot >= V_observed).mean())

    return dict(
        wrc_pvalue=wrc_pvalue,
        V_observed=V_observed,
        candidate_sharpe=cand_sharpe,
        best_pool_sharpe=pool_sharpes.max(),
        pool_sharpes=pool_sharpes,
        V_boot=V_boot
    )


print("Running White's Reality Check (1,000 strategies × 1,000 bootstrap) ...")
wrc = whites_reality_check(candidate_returns, random_pool, n_boot=1_000, rng_obj=rng)

print(f"\n── White's Reality Check Results ────────────────────────────────")
print(f"  Candidate Sharpe   : {wrc['candidate_sharpe']:+.3f}")
print(f"  Best pool Sharpe   : {wrc['best_pool_sharpe']:+.3f}")
print(f"  WRC test statistic : {wrc['V_observed']:+.3f}")
print(f"  WRC p-value        : {wrc['wrc_pvalue']:.4f}")
print(f"\n  Interpretation: {'Reject H0 — candidate has genuine edge (p < 0.05)' if wrc['wrc_pvalue'] < 0.05 else 'Fail to reject H0 — could be data snooping (p >= 0.05)'}")

Running White's Reality Check (1,000 strategies × 1,000 bootstrap) ...

── White's Reality Check Results ────────────────────────────────
  Candidate Sharpe   : -0.163
  Best pool Sharpe   : +2.004
  WRC test statistic : +2.166
  WRC p-value        : 0.2990

  Interpretation: Fail to reject H0 — could be data snooping (p >= 0.05)


In [12]:
# ── Visualise WRC bootstrap distribution ─────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=wrc['V_boot'],
    nbinsx=60,
    name='Bootstrap V* distribution',
    marker_color='rgba(100,149,237,0.6)',
    marker_line=dict(color='cornflowerblue', width=0.3)
))

fig.add_vline(
    x=wrc['V_observed'],
    line_color='#00d4aa', line_dash='dash',
    annotation_text=f"V_observed = {wrc['V_observed']:.2f}",
    annotation_position='top right'
)

fig.add_annotation(
    x=0.97, y=0.95,
    xref='paper', yref='paper',
    text=f"WRC p-value = {wrc['wrc_pvalue']:.4f}<br>{'✓ Reject H0 (p < 0.05)' if wrc['wrc_pvalue'] < 0.05 else '✗ Fail to reject H0 (p ≥ 0.05)'}",
    showarrow=False, align='right',
    bgcolor='rgba(0,0,0,0.65)', bordercolor='#555',
    font=dict(size=13, color='white')
)

fig.update_layout(
    title="White's Reality Check — Bootstrap Distribution of Max Excess Performance",
    xaxis_title='V* (max excess Sharpe of random pool vs. candidate)',
    yaxis_title='Frequency',
    template='plotly_dark',
    showlegend=False
)
fig.show()

In [13]:
# ── Distribution of pool Sharpe ratios vs. candidate ─────────────────────────
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=wrc['pool_sharpes'],
    nbinsx=60,
    histnorm='probability density',
    name='Random pool Sharpe dist.',
    marker_color='rgba(255,107,107,0.5)',
    marker_line=dict(color='#ff6b6b', width=0.3)
))

fig.add_vline(
    x=wrc['candidate_sharpe'],
    line_color='#00d4aa', line_width=2, line_dash='dot',
    annotation_text=f"Candidate: {wrc['candidate_sharpe']:.2f}",
    annotation_position='top right'
)

# Mark best random strategy
fig.add_vline(
    x=wrc['best_pool_sharpe'],
    line_color='lightsalmon', line_width=2, line_dash='dash',
    annotation_text=f"Best random: {wrc['best_pool_sharpe']:.2f}",
    annotation_position='top left'
)

fig.update_layout(
    title='Candidate Sharpe vs. Distribution of 1,000 Random Strategy Sharpes',
    xaxis_title='Annualised Sharpe Ratio',
    yaxis_title='Density',
    template='plotly_dark',
    showlegend=False
)
fig.show()

## 8  |  Metric Sensitivity Deep-Dive

Now we deliberately stress-test the metrics by varying **two scenario types** and observing how each metric responds:

- **Fat-tailed returns** (Student-t) — same mean/vol, heavy tails
- **Negatively skewed returns** (GEV-like) — crash-prone distribution

In [14]:
def scaled_t(mu, sig, df, n, rng_obj):
    """Student-t returns scaled to match target mean/vol."""
    raw = rng_obj.standard_t(df, size=n)
    raw = raw / raw.std(ddof=1) * sig + mu
    return raw

def skewed_normal(mu, sig, skew_param, n, rng_obj):
    """Skewed normal via scipy."""
    from scipy.stats import skewnorm
    raw = skewnorm.rvs(a=skew_param, size=n, random_state=int(rng_obj.integers(0, 9999)))
    raw = (raw - raw.mean()) / raw.std(ddof=1) * sig + mu
    return raw

scenarios = {
    'Normal (baseline)' : candidate_returns,
    'Fat tails (t=4)'   : scaled_t(MU_CANDIDATE, VOL_DAILY, df=4,  n=N_DAYS, rng_obj=rng),
    'Fat tails (t=2)'   : scaled_t(MU_CANDIDATE, VOL_DAILY, df=2,  n=N_DAYS, rng_obj=rng),
    'Neg. skew (-3)'    : skewed_normal(MU_CANDIDATE, VOL_DAILY, -3, N_DAYS, rng),
    'Neg. skew (-6)'    : skewed_normal(MU_CANDIDATE, VOL_DAILY, -6, N_DAYS, rng),
}

scenario_metrics = {name: compute_metrics(ret) for name, ret in scenarios.items()}

scenario_df = pd.DataFrame(scenario_metrics).T[
    ['sharpe', 'sortino', 'calmar', 'skewness', 'kurtosis', 'jb_pval', 't_pval', 'ann_return', 'max_dd']
].round(3)

print(scenario_df.to_string())

                   sharpe  sortino  calmar  skewness  kurtosis  jb_pval  t_pval  ann_return  max_dd
Normal (baseline)  -0.163   -0.271  -0.095     0.032    -0.086    0.833   0.611      -0.025  -0.267
Fat tails (t=4)     0.719    1.049   0.524    -0.235     2.205    0.000   0.107       0.114  -0.218
Fat tails (t=2)    -0.320   -0.376  -0.184    -0.528    14.324    0.000   0.710      -0.051  -0.276
Neg. skew (-3)      0.476    0.655   0.366    -0.669     0.282    0.000   0.205       0.076  -0.206
Neg. skew (-6)      0.476    0.601   0.484    -0.996     1.767    0.000   0.205       0.076  -0.156


In [15]:
# Normalise each metric to [0, 1] for radar / grouped bar comparison
compare_cols = ['sharpe', 'sortino', 'calmar', 'skewness', 'kurtosis']

norm_df = scenario_df[compare_cols].copy()
for c in compare_cols:
    col_min, col_max = norm_df[c].min(), norm_df[c].max()
    if col_max != col_min:
        norm_df[c] = (norm_df[c] - col_min) / (col_max - col_min)
    else:
        norm_df[c] = 0.5

colors = ['#00d4aa', 'cornflowerblue', '#6495ed', 'lightsalmon', '#ff6b6b']

fig = go.Figure()
for (scenario, row), color in zip(norm_df.iterrows(), colors):
    fig.add_trace(go.Scatterpolar(
        r=row.values.tolist() + [row.values[0]],
        theta=compare_cols + [compare_cols[0]],
        mode='lines+markers',
        name=scenario,
        line=dict(color=color)
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Metric Sensitivity Across Distribution Scenarios (normalised)',
    template='plotly_dark',
    legend=dict(x=1.05)
)
fig.show()

## 9  |  Summary & Conclusions

### Which metrics agree?
- **Sharpe, Sortino, and T-test p-value** are highly rank-correlated (all derived from mean/std). A strategy that ranks well on Sharpe almost always ranks well on Sortino and yields a low T-test p-value.
- **Calmar** broadly agrees with Sharpe on normally distributed returns but **diverges** when drawdowns become path-dependent (fat tails, negative skew) — it is less stable across scenarios.

### Which metrics add independent depth?
- **Skewness and Kurtosis** expose crash risk invisible to Sharpe. A strategy can have a stellar Sharpe but deeply negative skew and excess kurtosis — a known pattern in short-volatility, carry, or option-selling strategies.
- **Jarque-Bera** formalises the combined skew + kurtosis departure from normality. A low JB p-value is a warning that standard risk formulas (which assume normality) will understate tail losses.
- **Bootstrap CI for Sharpe** reveals estimation uncertainty. A Sharpe of 0.8 with a 95% CI of [−0.1, 1.7] is very different from one with CI [0.5, 1.1].

### White's Reality Check verdict
```
WRC p-value < 0.05  → Candidate has genuine alpha beyond the noise baseline.
WRC p-value ≥ 0.05  → Cannot rule out data snooping; the strategy may simply
                       be the lucky winner of a random search.
```

> **Key takeaway:** No single metric tells the full story. Using return-based metrics (Sharpe, Sortino, Calmar) together with distribution-shape metrics (skewness, kurtosis, JB) and a multiple-comparison correction (White's Reality Check) gives a much more complete and honest picture of whether a strategy has genuine alpha.